# Technology-trade HS transition pilot audit

This notebook reviews four native-classification cases:

- Lesotho: 2004 (HS1992) and 2005 (HS1996)
- Suriname: 2008 (HS1992) and 2009 (HS1996)

It makes no API calls. Run `python -m src.data.run_technology_transition_pilot` first.


In [1]:
from pathlib import Path
import pandas as pd


def find_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Repository root not found")

root = find_root()
output_dir = root / "outputs" / "tables" / "technology"
manifest = pd.read_csv(output_dir / "technology_transition_pilot_manifest.csv")
coverage = pd.read_csv(output_dir / "technology_transition_pilot_coverage.csv")
print("Repository root:", root)
print("Manifest rows:", len(manifest))
print("Coverage rows:", len(coverage))


Repository root: .
Manifest rows: 8
Coverage rows: 4


## 1. Request and cache outcomes


In [2]:
status_summary = (
    manifest.groupby(
        ["pilot_case", "expected_classification", "hs_revision", "anchor_iso3", "status"],
        dropna=False,
    )
    .agg(
        requested_hs6=("requested_hs6", "max"),
        returned_hs6=("returned_hs6", "max"),
        rows=("rows", "sum"),
    )
    .reset_index()
)
display(status_summary)
assert not manifest["status"].astype(str).str.startswith("error:").any()


,pilot_case,expected_classification,hs_revision,anchor_iso3,status,requested_hs6,returned_hs6,rows
0,LSO_2004,H0,HS1992,CHN,downloaded,90,2,2
1,LSO_2004,H0,HS1992,USA,downloaded,90,5,5
2,LSO_2005,H1,HS1996,CHN,downloaded_empty,110,0,0
3,LSO_2005,H1,HS1996,USA,downloaded,110,4,4
4,SUR_2008,H0,HS1992,CHN,downloaded,90,39,39
5,SUR_2008,H0,HS1992,USA,downloaded,90,50,50
6,SUR_2009,H1,HS1996,CHN,downloaded,110,37,37
7,SUR_2009,H1,HS1996,USA,downloaded,110,60,60


## 2. Coverage status


In [3]:
display(
    coverage[[
        "pilot_case", "reporter_iso", "year", "classification_code",
        "hs_revision", "technology_trade_status",
    ]]
)
assert coverage["technology_trade_status"].eq("available").all()


,pilot_case,reporter_iso,year,classification_code,hs_revision,technology_trade_status
0,LSO_2004,LSO,2004,H0,HS1992,available
1,LSO_2005,LSO,2005,H1,HS1996,available
2,SUR_2008,SUR,2008,H0,HS1992,available
3,SUR_2009,SUR,2009,H1,HS1996,available


## 3. Raw-data audit


In [5]:
from pandas.errors import EmptyDataError

raw_frames = []
raw_file_audit = []

for row in manifest.itertuples(index=False):
    path = root / row.raw_path

    try:
        frame = pd.read_csv(path, dtype={"cmdCode": str})
        file_status = "readable"
    except EmptyDataError:
        frame = pd.DataFrame()
        file_status = "empty_file"

    raw_file_audit.append(
        {
            "pilot_case": row.pilot_case,
            "anchor_iso3": row.anchor_iso3,
            "expected_classification": row.expected_classification,
            "hs_revision": row.hs_revision,
            "manifest_status": row.status,
            "raw_path": row.raw_path,
            "file_status": file_status,
            "rows": len(frame),
        }
    )

    if not frame.empty:
        frame["pilot_case"] = row.pilot_case
        frame["anchor_iso3"] = row.anchor_iso3
        frame["expected_classification"] = row.expected_classification
        frame["hs_revision"] = row.hs_revision
        raw_frames.append(frame)

raw_file_audit = pd.DataFrame(raw_file_audit)

display(raw_file_audit)

raw = (
    pd.concat(raw_frames, ignore_index=True)
    if raw_frames
    else pd.DataFrame()
)

print("Readable non-empty raw rows:", len(raw))
print(
    "Empty raw files:",
    raw_file_audit["file_status"].eq("empty_file").sum(),
)

if not raw.empty:
    print("Columns:", raw.columns.tolist())

,pilot_case,anchor_iso3,expected_classification,hs_revision,manifest_status,raw_path,file_status,rows
0,LSO_2004,CHN,H0,HS1992,downloaded,data/raw/un_comtrade/technology_trade/2004/LSO...,readable,2
1,LSO_2004,USA,H0,HS1992,downloaded,data/raw/un_comtrade/technology_trade/2004/LSO...,readable,5
2,LSO_2005,CHN,H1,HS1996,downloaded_empty,data/raw/un_comtrade/technology_trade/2005/LSO...,empty_file,0
3,LSO_2005,USA,H1,HS1996,downloaded,data/raw/un_comtrade/technology_trade/2005/LSO...,readable,4
4,SUR_2008,CHN,H0,HS1992,downloaded,data/raw/un_comtrade/technology_trade/2008/SUR...,readable,39
5,SUR_2008,USA,H0,HS1992,downloaded,data/raw/un_comtrade/technology_trade/2008/SUR...,readable,50
6,SUR_2009,CHN,H1,HS1996,downloaded,data/raw/un_comtrade/technology_trade/2009/SUR...,readable,37
7,SUR_2009,USA,H1,HS1996,downloaded,data/raw/un_comtrade/technology_trade/2009/SUR...,readable,60


Readable non-empty raw rows: 197
Empty raw files: 1
Columns: ['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate', 'pilot_case', 'anchor_iso3', 'expected_classification', 'hs_revision']


In [8]:
value_candidates = [
    "primaryValue",
    "TradeValue",
    "tradeValue",
    "fobvalue",
    "cifvalue",
]

if raw.empty:
    raw_summary = pd.DataFrame(
        columns=[
            "pilot_case",
            "expected_classification",
            "hs_revision",
            "anchor_iso3",
            "rows",
            "hs6_returned",
            "trade_value_usd",
        ]
    )
else:
    value_col = next(
        column
        for column in value_candidates
        if column in raw.columns
    )

    raw[value_col] = pd.to_numeric(
        raw[value_col],
        errors="coerce",
    )

    raw_summary = (
        raw.groupby(
            [
                "pilot_case",
                "expected_classification",
                "hs_revision",
                "anchor_iso3",
            ]
        )
        .agg(
            rows=(value_col, "size"),
            hs6_returned=("cmdCode", "nunique"),
            trade_value_usd=(value_col, "sum"),
        )
        .reset_index()
    )

expected_rows = manifest[
    [
        "pilot_case",
        "expected_classification",
        "hs_revision",
        "anchor_iso3",
        "status",
        "requested_hs6",
        "returned_hs6",
    ]
].copy()

raw_summary = expected_rows.merge(
    raw_summary,
    on=[
        "pilot_case",
        "expected_classification",
        "hs_revision",
        "anchor_iso3",
    ],
    how="left",
    validate="one_to_one",
)

raw_summary["rows"] = raw_summary["rows"].fillna(0).astype(int)
raw_summary["hs6_returned"] = (
    raw_summary["hs6_returned"].fillna(0).astype(int)
)
raw_summary["trade_value_usd"] = (
    raw_summary["trade_value_usd"].fillna(0.0)
)

display(raw_summary)

,pilot_case,expected_classification,hs_revision,anchor_iso3,status,requested_hs6,returned_hs6,rows,hs6_returned,trade_value_usd
0,LSO_2004,H0,HS1992,CHN,downloaded,90,2,2,2,1.682634e+04
1,LSO_2004,H0,HS1992,USA,downloaded,90,5,5,5,1.369114e+03
2,LSO_2005,H1,HS1996,CHN,downloaded_empty,110,0,0,0,0.000000e+00
3,LSO_2005,H1,HS1996,USA,downloaded,110,4,4,4,6.172040e+02
4,SUR_2008,H0,HS1992,CHN,downloaded,90,39,39,39,1.522022e+06
5,SUR_2008,H0,HS1992,USA,downloaded,90,50,50,50,2.831427e+07
6,SUR_2009,H1,HS1996,CHN,downloaded,110,37,37,37,4.514063e+06
7,SUR_2009,H1,HS1996,USA,downloaded,110,60,60,60,2.048775e+07


## 4. Transition comparison


In [9]:
totals = raw_summary.pivot_table(
    index=["pilot_case", "expected_classification", "hs_revision"],
    columns="anchor_iso3",
    values="trade_value_usd",
    aggfunc="sum",
).reset_index()
for anchor in ["CHN", "USA"]:
    if anchor not in totals.columns:
        totals[anchor] = 0.0
totals["orientation_score"] = (
    (totals["CHN"] - totals["USA"])
    / (totals["CHN"] + totals["USA"]).replace(0, pd.NA)
)
totals["reporter_iso"] = totals["pilot_case"].str[:3]
totals["year"] = totals["pilot_case"].str[-4:].astype(int)
totals = totals.sort_values(["reporter_iso", "year"])
totals["orientation_change"] = totals.groupby("reporter_iso")["orientation_score"].diff()
display(
    totals[[
        "reporter_iso", "year", "expected_classification", "hs_revision",
        "CHN", "USA", "orientation_score", "orientation_change",
    ]]
)


anchor_iso3,reporter_iso,year,expected_classification,hs_revision,CHN,USA,orientation_score,orientation_change
0,LSO,2004,H0,HS1992,16826.337,1.369114e+03,0.849510,NaN
1,LSO,2005,H1,HS1996,0.000,6.172040e+02,-1.000000,-1.849510
2,SUR,2008,H0,HS1992,1522022.000,2.831427e+07,-0.897975,NaN
3,SUR,2009,H1,HS1996,4514063.000,2.048775e+07,-0.638901,0.259074


## Decision checkpoint

Proceed to global collection if:

- neither anchor has an error status;
- all four reporter-years are available;
- expected H0/H1 metadata are preserved;
- returned product coverage is plausible and neither anchor is unexpectedly empty;
- transition changes do not show an obvious extraction failure.

Classification-transition effects will be assessed systematically after the global panel is assembled.
